In [ ]:
import requests
from IPython.display import JSON
import json
from requests.auth import HTTPBasicAuth

basic = HTTPBasicAuth('elastic', 'PGUJYTxo')

def es_get(url: str, body: dict = None):
    result = requests.get(f"http://localhost:9200/{url}", json=body, auth=basic).json()
    return JSON(result, expanded=True)


def es_post(url: str, body: dict = None):
    result = requests.post(f"http://localhost:9200/{url}", json=body, auth=basic).json()
    return JSON(result, expanded=True)


def es_put(url: str, body: dict):
    result = requests.put(f"http://localhost:9200/{url}", json=body, auth=basic).json()
    return JSON(result, expanded=True)


def es_bulk(lines: list):
    result = requests.put(f"http://localhost:9200/_bulk", data=lines,
                          headers={"Content-Type": "application/json"}, auth=basic).json()
    return JSON(result, expanded=True)

In [ ]:
es_post("books*/_delete_by_query",
        {
            "query": {
                "match_all": {}
            }
        })

# Intro

[Elasticsearch](https://www.elastic.co/elasticsearch) is based on [Apache Lucene](https://lucene.apache.org/) which provides the basic building blocks for search, like an inverted index and TF-IDF scoring. Elasticsearch adds on top of that a REST API, index management facilities and additional features like aggregations, query types and suggesters. Additionally, the backing company Elastic provides a whole ecosystem for ingesting and analyzing many different types of documents like log messages.

Besides full-text search capabilities, Elasticsearch acts as a NoSQL database. Documents are organized in indices, and are represented by JSON structures. Document fields can be mapped to different types like text, number, boolean or date. Text fields can be analyzed according to the full-text search requirements.

# Indexing

Add one document.

In [ ]:
es_post("books/_doc",
        {
            "name": "Snow Crash",
            "author": "Neal Stephenson",
            "release_date": "1992-06-01",
            "page_count": 470
        })

Add multiple documents.

In [ ]:
es_bulk("""
{ "index" : { "_index" : "books" } }
{"name": "Revelation Space", "author": "Alastair Reynolds", "release_date": "2000-03-15", "page_count": 585}
{ "index" : { "_index" : "books" } }
{"name": "1984", "author": "George Orwell", "release_date": "1985-06-01", "page_count": 328}
{ "index" : { "_index" : "books" } }
{"name": "Fahrenheit 451", "author": "Ray Bradbury", "release_date": "1953-10-15", "page_count": 227}
{ "index" : { "_index" : "books" } }
{"name": "Brave New World", "author": "Aldous Huxley", "release_date": "1932-06-01", "page_count": 268}
{ "index" : { "_index" : "books" } }
{"name": "The Handmaids Tale", "author": "Margaret Atwood", "release_date": "1985-06-01", "page_count": 311}
""")

Added documents are not immediately visible after indexing. To make them visible, you can force a refresh.

In [ ]:
es_post("books/_refresh")

# Exploring the index with Kibana

1. Open the [Kibana Discovery view](http://127.0.0.1:5601/app/discover).
2. Create a new data view for the index pattern `books` without a timestamp field.

You can now try different things:

* View details of a documents.
* Sort documents.
* Select which fields to show in the list of documents.
* Filters documents using KQL.
* Visualize field statistics.

# Searching

Just return all documents in the index.

In [ ]:
es_get("books/_search")

Let's run a simple query to `match` the term `brave` in the field `name`

In [ ]:
es_get("books/_search",
       {
           "query": {
               "match": {
                   "name": "brave"
               }
           }
       })

You can also search across multiple fields. By default, at least one search term has to match (`OR`).

In [ ]:
es_get("books/_search",
       {
           "query": {
               "multi_match": {
                   "query": "brave george",
                   "fields": ["name", "author"]
               }
           }
       })

You can also require that all search terms must match (`AND`). Use `cross_fields` type, otherwise all search terms must be in the same field.

In [ ]:
es_get("books/_search",
       {
           "query": {
               "multi_match": {
                   "query": "brave george",
                   "fields": ["name", "author"],
                   "operator": "and",
                   "type": "cross_fields"
               }
           }
       })

In [ ]:
es_get("books/_search",
       {
           "query": {
               "multi_match": {
                   "query": "brave aldous",
                   "fields": ["name", "author"],
                   "operator": "and",
                   "type": "cross_fields"
               }
           }
       })

Fields can also be weighted.

In [ ]:
es_get("books/_search",
       {
           "query": {
               "multi_match": {
                   "query": "brave aldous",
                   "fields": ["name^5", "author^1"],
                   "operator": "and",
                   "type": "cross_fields"
               }
           }
       })

Alternative, you can use a compound query.

In [ ]:
es_get("books/_search",
       {
           "query": {
               "bool": {
                   "must": [
                       {
                           "multi_match": {
                               "query": "brave",
                               "fields": ["name", "author"],
                               "type": "best_fields"
                           }
                       },
                       {
                           "multi_match": {
                               "query": "aldous",
                               "fields": ["name", "author"],
                               "type": "best_fields"
                           }
                       }
                   ]
               }
           }
       })

The compound query also supports `should` for optional matches that are not required but boost the score when matched.

In [ ]:
es_get("books/_search",
       {
           "query": {
               "bool": {
                   "should": [
                       {
                           "multi_match": {
                               "query": "brave",
                               "fields": ["name", "author"],
                               "type": "best_fields"
                           }
                       },
                       {
                           "multi_match": {
                               "query": "george",
                               "fields": ["name", "author"],
                               "type": "best_fields"
                           }
                       }
                   ]
               }
           }
       })

In [ ]:
es_get("books/_search",
       {
           "suggest": {
               "text": "brve new word",
               "my_phrase_suggestion": {
                   "phrase": {
                       "field": "name",
                       "size": 1,
                       "gram_size": 3,
                       "direct_generator": [{
                           "field": "name",
                           "suggest_mode": "always",
                           "min_word_length": 1
                       }]
                   }
               }
           }
       })

# Index schema

What does the field mapping look like?

In [ ]:
es_get("books/_mapping")

Elasticsearch automatically infers suitable field types based on the field contents during indexing e.g. `name` is `text`, `page_count` is `long` and `release_date` is `date`.

`text` is the default field for (English) full-text search.

Automatically mapped `text` fields have a sub-field named `.keyword` that matches the entire field value.

In [ ]:
es_get("books/_search",
       {
           "query": {
               "match": {
                   "name.keyword": "brave"
               }
           }
       })

In [ ]:
es_get("books/_search",
       {
           "query": {
               "match": {
                   "name.keyword": "Brave New World"
               }
           }
       })

# Analyzer

In [ ]:
es_post("_analyze",
        {
            "analyzer": "standard",
            "text": "The brave new Worlds!"
        })

Use `english` for stemming and stop word removal.

In [ ]:
es_post("_analyze",
        {
            "analyzer": "english",
            "text": "The brave new Worlds!"
        })

Or `german` for German-specific analysis.

In [ ]:
es_post("_analyze",
        {
            "analyzer": "german",
            "text": "Die tapferen neuen Welten!"}
        )

Use explicit mapping to define fields.

In [ ]:
es_put("books_de",
       {
           "mappings": {
               "properties": {
                   "name": {"type": "text", "analyzer": "german"},
                   "author": {"type": "text"},
                   "release_date": {"type": "date"},
                   "page_count": {"type": "long"}
               }
           }
       })

In [ ]:
es_post(
    "books_de/_doc",
    {
        "name": "Die tapferen neuen Welten!",
        "author": "Aldous Huxley",
        "release_date": "1932-06-01",
        "page_count": 268,
    },
)

In [ ]:
es_get("books_de/_search",
       {
           "query": {
               "match": {
                   "name": "welt"
               }
           }
       })

# Return fields

In [ ]:
es_get("books/_search",
       {
           "fields": ["name", "release_date"],
           "_source": False
       })

# Pagination

In [ ]:
es_get("books/_search",
       {
           "from": 1,
           "size": 2
       })

# Sorting

In [ ]:
es_get("books/_search",
       {
           "sort": [
               {
                   "release_date": {
                       "order": "asc"
                   }
               }
           ]
       })

In [ ]:
es_get("books/_search",
       {
           "sort": [
               {
                   "name.keyword": {
                       "order": "asc"
                   }
               }
           ]
       })

# Aggregations

Let's add a `genre` field to the documents, and an `_id` so documents can be updated if needed.

In [ ]:
es_post("books/_delete_by_query",
        {
            "query": {
                "match_all": {}
            }
        })

es_bulk("""
{"index": {"_index": "books", "_id": 1}}
{"name": "Revelation Space", "author": "Alastair Reynolds", "release_date": "2000-03-15", "page_count": 585, "genre": "Science Fiction"}
{"index": {"_index": "books", "_id": 2}}
{"name": "1984", "author": "George Orwell", "release_date": "1985-06-01", "page_count": 328, "genre": "Dystopian"}
{"index": {"_index": "books", "_id": 3}}
{"name": "Fahrenheit 451", "author": "Ray Bradbury", "release_date": "1953-10-15", "page_count": 227, "genre": "Dystopian"}
{"index": {"_index": "books", "_id": 4}}
{"name": "Brave New World", "author": "Aldous Huxley", "release_date": "1932-06-01", "page_count": 268, "genre": "Dystopian"}
{"index": {"_index": "books", "_id": 5}}
{"name": "The Handmaids Tale", "author": "Margaret Atwood", "release_date": "1985-06-01", "page_count": 311, "genre": "Dystopian"}
{"index": {"_index": "books", "_id": 6}}
{"name": "The Hobbit", "author": "J.R.R. Tolkien", "release_date": "1937-09-21", "page_count": 310, "genre": "Fantasy"}
{"index": {"_index": "books", "_id": 7}}
{"name": "A Game of Thrones", "author": "George R.R. Martin", "release_date": "1996-08-06", "page_count": 694, "genre": "Fantasy"}
{"index": {"_index": "books", "_id": 8}}
{"name": "Hyperion", "author": "Dan Simmons", "release_date": "1989-05-26", "page_count": 482, "genre": "Science Fiction"}
{"index": {"_index": "books", "_id": 9}}
{"name": "Dune", "author": "Frank Herbert", "release_date": "1965-08-01", "page_count": 412, "genre": "Science Fiction"}
""")

es_post("books/_refresh")

In [ ]:
es_get("books/_search",
       {
           "size": 0,
           "aggs": {
               "genres": {
                   "terms": {
                       "field": "genre.keyword"
                   }
               }
           }
       })

In [ ]:
es_get("books/_search",
       {
           "size": 0,
           "aggs": {
               "page_ranges": {
                   "range": {
                       "field": "page_count",
                       "ranges": [
                           {"from": 0, "to": 100},
                           {"from": 101, "to": 200},
                           {"from": 201, "to": 300},
                           {"from": 301, "to": 400},
                           {"from": 401, "to": 500}
                       ]
                   }
               }
           }
       })

# Filters

In [ ]:
es_get("books/_search",
       {
           "query": {
               "term": {
                   "genre.keyword": {
                       "value": "Dystopian"
                   }
               }
           }
       })

In [ ]:
es_get("books/_search",
       {
           "query": {
               "range": {
                   "page_count": {
                       "gte": 500
                   }
               }
           }
       })

Filters can also be expressed as compound queries.

In [ ]:
es_get("books/_search",
       {
           "query": {
               "bool": {
                   "filter": [
                       {
                           "term": {
                               "genre.keyword": {
                                   "value": "Dystopian"
                               }
                           }
                       }
                   ]
               }
           }
       })

And combined with optional search terms for boosting matching results.

In [ ]:
es_get("books/_search",
       {
           "query": {
               "bool": {
                   "filter": [
                       {
                           "term": {
                               "genre.keyword": {
                                   "value": "Dystopian"
                               }
                           }
                       }
                   ],
                   "should": [
                       {
                           "multi_match": {
                               "query": "brave",
                               "fields": ["name", "author"],
                               "type": "best_fields"
                           }
                       }
                   ],
               }
           }
       })

# Debugging

In [ ]:
es_get("books/_search",
       {
           "query": {
               "multi_match": {
                   "query": "brave george",
                   "fields": ["name", "author"],
               }
           },
           "explain": True
       })

In [ ]:
es_get("books/_explain/4",
       {
           "query": {
               "match": {
                   "name": "world"
               }
           }
       })

In [ ]:
es_get("books/_explain/1",
       {
           "query": {
               "match": {
                   "name": "world"
               }
           }
       })